**Olivia Rodrigo GUTS World Tour: price data collection**

Same process as the Unraveled Tour notebook, but everything is named and saved separately (`olivia_rodrigo_guts_data` folder, `guts_` file names), so none of the current tour data gets overwritten. Price history is only downloaded up to the day of each show, and the final CSV has the same columns as the Unraveled Tour final file.

### Setup

In [ ]:
import os
import time
import glob
import requests
import pandas as pd
from google.colab import drive
from google.colab import userdata

drive.mount('/content/drive')

# Separate GUTS folders, so nothing from the Unraveled Tour data gets overwritten.
GUTS_OUTPUT_DIR = "/content/drive/MyDrive/olivia_rodrigo_guts_data"
GUTS_RAW_DIR = os.path.join(GUTS_OUTPUT_DIR, "daily_by_event")
os.makedirs(GUTS_RAW_DIR, exist_ok=True)

API_KEY = userdata.get("api_key_seat")
BASE_URL = "https://seatdata.io/api/v1"
HEADERS = {"Authorization": f"Bearer {API_KEY}"}

# GUTS World Tour window (first show to last show, including the 2025 "Spilled" stadium dates).
GUTS_TOUR_START = "2024-02-23"
GUTS_TOUR_END = "2025-07-01"

# The tour was announced in September 2023, so no price history can exist before this.
GUTS_HISTORY_START = "2023-09-01"


Mounted at /content/drive


**Find the GUTS tour id.** Searching a few different months of the tour shows which `std_event_id` values come back. The Unraveled Tour was 10384. If GUTS has more than one id (for example the stadium dates), run the next cell once per id.

In [ ]:
# Search a few different months of the GUTS tour and see which tour ids (std_event_id) come back.
# The Unraveled Tour was 10384. GUTS may have one id, or a separate id for the stadium dates.
guts_probe_months = ["2024-03", "2024-08", "2024-10", "2025-06"]

probe_frames = []
for month in guts_probe_months:
    response = requests.get(
        f"{BASE_URL}/events/search",
        headers=HEADERS,
        params={"event_name": "Olivia Rodrigo", "event_date": month, "historical": "true", "limit": 20},
        timeout=30,
    )
    response.raise_for_status()
    probe_frames.append(pd.DataFrame(response.json().get("data", [])))

guts_probe_df = pd.concat(probe_frames, ignore_index=True)
guts_probe_df["event_date"] = pd.to_datetime(guts_probe_df["event_date"])

display(
    guts_probe_df.groupby("std_event_id").agg(
        shows_seen=("event_id", "nunique"),
        first_show=("event_date", "min"),
        last_show=("event_date", "max"),
        example_venue=("venue_name", "first"),
    )
)


,shows_seen,first_show,last_show,example_venue
std_event_id,,,,
7736,1,2024-03-29,2024-03-29,Hall 224
8063,1,2024-03-01,2024-03-01,Mr. Smalls Theatre
10384,42,2024-03-01,2025-06-30,American Airlines Center
26351,2,2025-06-12,2025-06-14,The Bonnaroo Farm
34980,1,2025-06-27,2025-06-27,Hyde Park
35016,2,2025-06-20,2025-06-21,Megaland Evenemententerrein
35061,4,2025-06-06,2025-06-07,Corona Park Flushing Meadows
42950,1,2025-06-28,2025-06-28,Roskilde Festival
84714,1,2025-06-27,2025-06-27,The Phoenix


The GUTS tour had about 102 dates, it's clear the even ID 10381 containts the majority of them, so I will start by exploring that one. The dates also match closely the beginning and end of the tour.

In [ ]:
# Fill this in from the table above (the id whose shows fall inside the GUTS tour dates).
GUTS_STD_TOUR_ID = 10384

if GUTS_STD_TOUR_ID is None:
    raise ValueError("Set GUTS_STD_TOUR_ID using the table in the previous cell.")

guts_events = []
params = {"std_event_id": GUTS_STD_TOUR_ID, "historical": "true", "limit": 200}

while True:
    resp = requests.get(f"{BASE_URL}/events/search", headers=HEADERS, params=params, timeout=30)
    resp.raise_for_status()
    result = resp.json()
    guts_events.extend(result.get("data", []))
    cursor = result.get("next_cursor")
    if not cursor:
        break
    params["starting_after"] = cursor

guts_tour_df = pd.DataFrame(guts_events)
guts_tour_df["event_date"] = pd.to_datetime(guts_tour_df["event_date"])

# Keep only shows inside the GUTS tour window.
guts_tour_df = (
    guts_tour_df[guts_tour_df["event_date"].between(GUTS_TOUR_START, GUTS_TOUR_END)]
    .drop_duplicates(subset=["event_id"])
    .sort_values("event_date")
    .reset_index(drop=True)
)

print(f"Total unique GUTS tour dates discovered in API: {len(guts_tour_df)}")
display(guts_tour_df[["event_id", "event_date", "venue_name", "venue_city"]].head(5))

# How much price history does SeatData have for each show? (The Unraveled Tour has about 184 days.)
# This tells us BEFORE spending money whether the old shows are worth downloading.
if "first_seen_date" in guts_tour_df.columns:
    guts_tour_df["first_seen_date"] = pd.to_datetime(guts_tour_df["first_seen_date"])
    guts_tour_df["days_of_history"] = (guts_tour_df["event_date"] - guts_tour_df["first_seen_date"]).dt.days
    print("\nDays of price history before each show:")
    display(guts_tour_df["days_of_history"].describe().round(0))


Total unique GUTS tour dates discovered in API: 93


,event_id,event_date,venue_name,venue_city
0,613100,2024-02-23,Acrisure Arena at Greater Palm Springs,Palm Desert
1,613364,2024-02-24,Footprint Center,Phoenix
2,613101,2024-02-27,Toyota Center,Houston
3,613113,2024-02-28,Moody Center ATX,Austin
4,613870,2024-03-01,American Airlines Center,Dallas



Days of price history before each show:


,days_of_history
count,93.0
mean,175.0
std,109.0
min,6.0
25%,114.0
50%,163.0
75%,203.0
max,536.0


**Looking at one show first** (only up to the show date), and confirm the same columns come back as for the Unraveled Tour.

In [ ]:
# Look at one show, only up to the day of the event, and check we get the same columns as the Unraveled data.
EXPECTED_STATS_COLUMNS = [
    "timestamp", "total_listings_all", "total_listings_active", "listing_fill_rate",
    "avg_price", "median_price", "get_in", "get_in_qty2plus", "zones",
]

sample_event = guts_tour_df.iloc[0]
sample_event_id = int(sample_event["event_id"])
sample_resp = requests.get(
    f"{BASE_URL}/events/{sample_event_id}/stats",
    headers=HEADERS,
    params={"start_date": GUTS_HISTORY_START,
            "end_date": sample_event["event_date"].strftime("%Y-%m-%d"),
            "limit": 10},
    timeout=30,
)
sample_resp.raise_for_status()
sample_stats = pd.DataFrame(sample_resp.json().get("data", []))
sample_stats["timestamp"] = pd.to_datetime(sample_stats["timestamp"])

print(f"Sample data retrieved for Event {sample_event_id}:")
display(sample_stats[["timestamp", "avg_price", "median_price", "get_in"]].head(5))

missing_columns = [c for c in EXPECTED_STATS_COLUMNS if c not in sample_stats.columns]
print("Columns missing compared to the Unraveled Tour data:", missing_columns if missing_columns else "none")


Sample data retrieved for Event 613100:


,timestamp,avg_price,median_price,get_in
0,2024-02-22 19:13:24+00:00,1517.58,620.800,252.20
1,2024-02-22 13:13:36+00:00,1465.13,601.975,272.57
2,2024-02-22 07:11:52+00:00,1434.57,630.000,278.39
3,2024-02-21 19:12:58+00:00,1430.54,637.515,260.93
4,2024-02-21 13:12:44+00:00,1426.10,651.315,253.17


Columns missing compared to the Unraveled Tour data: none


**Because API requests cost money, each show is downloaded once and saved to Drive.** Prices are only requested up to the day of the show.

In [ ]:
# Same columns as the Unraveled Tour dataset (this list is ordered, the metadata columns come first).
KEEP_COLUMNS = ["event_id", "event_name", "event_date", "venue"] + EXPECTED_STATS_COLUMNS


def download_guts_event_snapshots(event_id, event_name=None, event_date=None, venue=None):
    """Download every snapshot for one GUTS show, from the announcement up to the day of the show.
    Saved to Drive once, so running this again never pays for the same show twice."""
    filename = os.path.join(GUTS_RAW_DIR, f"guts_event_{event_id}_all_snapshots.csv")
    if os.path.exists(filename):
        return pd.read_csv(filename)

    show_date = pd.Timestamp(event_date)
    url = f"{BASE_URL}/events/{event_id}/stats"
    params = {"start_date": GUTS_HISTORY_START,
              "end_date": show_date.strftime("%Y-%m-%d"),   # nothing after the show
              "limit": 200}
    snapshots = []

    while True:
        response = requests.get(url, headers=HEADERS, params=params, timeout=60)
        response.raise_for_status()
        result = response.json()
        snapshots.extend(result.get("data", []))
        cursor = result.get("next_cursor")
        if not cursor:
            break
        params["starting_after"] = cursor
        time.sleep(0.1)

    if not snapshots:
        return None

    raw = pd.DataFrame(snapshots)
    raw["timestamp"] = pd.to_datetime(raw["timestamp"], utc=True)
    raw = raw.sort_values("timestamp", ascending=False)

    # Safety net: drop anything stamped after the show day, whatever the API returned.
    raw = raw[raw["timestamp"].dt.date <= show_date.date()]

    raw["event_id"] = event_id
    raw["event_name"] = event_name
    raw["event_date"] = show_date.strftime("%Y-%m-%d")
    raw["venue"] = venue

    raw = raw[[c for c in KEEP_COLUMNS if c in raw.columns]]
    raw.to_csv(filename, index=False)
    return raw


**Download the shows.** Keep `NUM_GUTS_EVENTS_TO_DOWNLOAD = 1` while testing, then raise it to `len(guts_tour_df)` to get everything. Shows already saved on Drive are skipped, so nothing is paid for twice.

In [ ]:
# Keep this small while testing, then raise it (or set it to len(guts_tour_df)) to download everything.
NUM_GUTS_EVENTS_TO_DOWNLOAD = len(guts_tour_df)

guts_events_to_process = guts_tour_df.head(NUM_GUTS_EVENTS_TO_DOWNLOAD)

already_saved = sum(
    os.path.exists(os.path.join(GUTS_RAW_DIR, f"guts_event_{int(e)}_all_snapshots.csv"))
    for e in guts_events_to_process["event_id"]
)
print(f"{len(guts_events_to_process)} events selected, {already_saved} already saved on Drive "
      f"(so {len(guts_events_to_process) - already_saved} will use the API).")

for position, (_, event) in enumerate(guts_events_to_process.iterrows(), start=1):
    download_guts_event_snapshots(
        event_id=int(event["event_id"]),
        event_name=event.get("event_name"),
        event_date=event.get("event_date"),
        venue=event.get("venue_name"),
    )
    if position % 10 == 0 or position == len(guts_events_to_process):
        print(f"Progress: {position}/{len(guts_events_to_process)} events processed and saved to Drive.")


93 events selected, 0 already saved on Drive (so 93 will use the API).
Progress: 10/93 events processed and saved to Drive.
Progress: 20/93 events processed and saved to Drive.
Progress: 30/93 events processed and saved to Drive.
Progress: 40/93 events processed and saved to Drive.
Progress: 50/93 events processed and saved to Drive.
Progress: 60/93 events processed and saved to Drive.
Progress: 70/93 events processed and saved to Drive.
Progress: 80/93 events processed and saved to Drive.
Progress: 90/93 events processed and saved to Drive.
Progress: 93/93 events processed and saved to Drive.


**Consolidate into one dataset** with the same rules and column order as the Unraveled Tour file.

In [ ]:
guts_csv_files = glob.glob(os.path.join(GUTS_RAW_DIR, "guts_event_*_all_snapshots.csv"))

guts_frames = []
for file_path in guts_csv_files:
    snapshots = pd.read_csv(file_path)
    snapshots["timestamp"] = pd.to_datetime(snapshots["timestamp"], utc=True)
    snapshots["date"] = snapshots["timestamp"].dt.date

    # Keep the latest observation per calendar date (same rule as the Unraveled Tour data).
    daily = snapshots.sort_values("timestamp", ascending=False).drop_duplicates("date", keep="first")
    guts_frames.append(daily)

guts_master_df = pd.concat(guts_frames, ignore_index=True)
guts_master_df["date"] = pd.to_datetime(guts_master_df["date"])

# Only days up to and including the show.
guts_master_df = guts_master_df[guts_master_df["date"] <= pd.to_datetime(guts_master_df["event_date"])]
guts_master_df = guts_master_df.sort_values(["event_id", "date"]).reset_index(drop=True)

# Same column order as the Unraveled Tour final file, so the model notebook can load either one.
FINAL_COLUMNS = [
    "timestamp", "total_listings_all", "total_listings_active", "listing_fill_rate", "avg_price",
    "median_price", "get_in", "get_in_qty2plus", "zones", "date", "event_id", "event_name",
    "event_date", "venue",
]
guts_master_df = guts_master_df[[c for c in FINAL_COLUMNS if c in guts_master_df.columns]]

guts_master_path = os.path.join(GUTS_OUTPUT_DIR, "olivia_rodrigo_guts_all_daily_prices.csv")
guts_master_df.to_csv(guts_master_path, index=False)

print("Consolidation complete!")
print(f"Total unique events gathered: {guts_master_df['event_id'].nunique()}")
print(f"Total pricing observations: {len(guts_master_df)}")
print("Columns missing vs Unraveled schema:", [c for c in FINAL_COLUMNS if c not in guts_master_df.columns] or "none")

# Quick health check: how many days of history per show, and does each one reach the show day?
health = guts_master_df.groupby("event_id").agg(
    event_date=("event_date", "first"), first_day=("date", "min"), last_day=("date", "max"), days=("date", "nunique"))
health["last_day_gap"] = (pd.to_datetime(health["event_date"]) - health["last_day"]).dt.days
display(health.describe().round(1))
display(guts_master_df.head(10))


Consolidation complete!
Total unique events gathered: 93
Total pricing observations: 8167
Columns missing vs Unraveled schema: none


,first_day,last_day,days,last_day_gap
count,93,93,93.0,93.0
mean,2024-03-21 03:36:46.451612928,2024-06-23 00:15:29.032258048,87.8,1.0
min,2024-02-12 00:00:00,2024-02-22 00:00:00,6.0,1.0
25%,2024-02-12 00:00:00,2024-04-01 00:00:00,36.0,1.0
50%,2024-02-12 00:00:00,2024-06-10 00:00:00,90.0,1.0
75%,2024-05-15 00:00:00,2024-08-13 00:00:00,126.0,1.0
max,2024-06-17 00:00:00,2025-06-30 00:00:00,187.0,1.0
std,NaN,NaN,57.6,0.0


,timestamp,total_listings_all,total_listings_active,listing_fill_rate,avg_price,median_price,get_in,get_in_qty2plus,zones,date,event_id,event_name,event_date,venue
0,2024-02-12 16:44:51+00:00,1772,333,0.187923,2730.11,855.000,315.00,432.00,"[{'zone_name': 'Club', 'avg_price': 2700.0, 'm...",2024-02-12,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
1,2024-02-13 19:13:01+00:00,1839,350,0.190321,2678.41,862.920,346.50,422.10,"[{'zone_name': 'Club', 'avg_price': 1800.0, 'm...",2024-02-13,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
2,2024-02-14 19:14:10+00:00,1861,343,0.184310,2709.68,852.750,346.50,450.00,"[{'zone_name': 'Club', 'avg_price': 1800.0, 'm...",2024-02-14,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
3,2024-02-15 19:14:42+00:00,1911,359,0.187860,2591.30,810.000,315.00,427.50,"[{'zone_name': 'Club', 'avg_price': 1800.0, 'm...",2024-02-15,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
4,2024-02-16 19:14:23+00:00,1977,364,0.184117,2548.51,775.515,358.83,358.90,"[{'zone_name': 'Bar', 'avg_price': 383.219, 'm...",2024-02-16,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
5,2024-02-17 19:14:12+00:00,2002,356,0.177822,2572.52,763.225,315.00,343.43,"[{'zone_name': 'Bar', 'avg_price': 366.728, 'm...",2024-02-17,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
6,2024-02-18 19:12:00+00:00,2052,336,0.163743,2651.96,774.410,321.68,321.68,"[{'zone_name': 'Bar', 'avg_price': 358.074, 'm...",2024-02-18,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
7,2024-02-19 19:11:23+00:00,2099,327,0.155788,2293.29,721.110,301.50,301.50,"[{'zone_name': 'Bar', 'avg_price': 331.698, 'm...",2024-02-19,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
8,2024-02-20 19:11:10+00:00,2148,327,0.152235,1830.52,720.000,268.45,268.45,"[{'zone_name': 'Bar', 'avg_price': 324.783, 'm...",2024-02-20,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
9,2024-02-21 19:12:58+00:00,2283,365,0.159877,1430.54,637.515,260.93,260.93,"[{'zone_name': 'Bar', 'avg_price': 306.714, 'm...",2024-02-21,613100,Olivia Rodrigo,2024-02-23,Acrisure Arena at Greater Palm Springs
